# 01 Data Splitting

## Purpose

This notebook creates the train, validation, and test splits for the rebuilt workflow.

## Input

- `MyDrive/ProjectRoot2/data/raw/raw_glycans_dataset_no_aldi.txt`

## Outputs

- `MyDrive/ProjectRoot2/data/splits/train.txt`
- `MyDrive/ProjectRoot2/data/splits/val.txt`
- `MyDrive/ProjectRoot2/data/splits/test.txt`
- `MyDrive/ProjectRoot2/data/splits/split_summary.csv`

## Notes to myself

This is where I lock in the split for the rebuilt workflow. I want it written to disk once, checked quickly, and then reused by the later notebooks.

## Setup note

Same pattern as notebook `00`.

- code stays in GitHub
- data and outputs stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

# Mount Drive so the notebook can read the raw file and write the split files.
drive.mount('/content/drive')

# Define the rebuilt GitHub repo.
GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta-sandbox2'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone the repo if needed. Otherwise update the existing clone in this runtime.
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

# Add the repo to the Python path so src/ imports work.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

## Path setup

Here I define the raw input file and the split output folder. I am using a dedicated `data/splits/` folder so the later notebooks do not have to guess where these files live.

In [ ]:
# ==============================================================================
# 1. DEFINE THE INPUT AND OUTPUT PATHS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'raw_glycans_dataset_no_aldi.txt')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')

os.makedirs(SPLITS_DIR, exist_ok=True)

print('Raw data path:')
print(RAW_DATA_PATH)
print('\nSplit output directory:')
print(SPLITS_DIR)

if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f'Raw dataset not found: {RAW_DATA_PATH}')

## Create the split files

I am using the helper in `src/data_utils.py` so the split logic lives in one place. The split is fixed by the random seed inside that helper, which is what I want for reproducibility.

In [ ]:
# ==============================================================================
# 2. SPLIT THE DATASET INTO TRAIN, VALIDATION, AND TEST SETS
# ==============================================================================
from src.data_utils import split_and_save_data

print('Starting data split...')
split_and_save_data(RAW_DATA_PATH, SPLITS_DIR)

## Quick verification

I want a short check here so I can confirm the files exist, the counts look reasonable, and the split did not silently go somewhere unexpected.

In [ ]:
# ==============================================================================
# 3. VERIFY THE SPLIT FILES AND SUMMARIZE THEM
# ==============================================================================
import pandas as pd

train_path = os.path.join(SPLITS_DIR, 'train.txt')
val_path = os.path.join(SPLITS_DIR, 'val.txt')
test_path = os.path.join(SPLITS_DIR, 'test.txt')

for split_path in [train_path, val_path, test_path]:
    if not os.path.exists(split_path):
        raise FileNotFoundError(f'Split file not found: {split_path}')

def load_sequences(path):
    with open(path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file if line.strip()]

train_sequences = load_sequences(train_path)
val_sequences = load_sequences(val_path)
test_sequences = load_sequences(test_path)

split_summary = pd.DataFrame(
    {
        'split': ['train', 'val', 'test'],
        'num_sequences': [
            len(train_sequences),
            len(val_sequences),
            len(test_sequences),
        ],
    }
)

display(split_summary)

preview_rows = []
for split_name, sequences in [
    ('train', train_sequences),
    ('val', val_sequences),
    ('test', test_sequences),
]:
    preview_rows.append(
        {
            'split': split_name,
            'example_sequence': sequences[0] if sequences else '',
            'char_length': len(sequences[0]) if sequences else 0,
        }
    )

split_preview = pd.DataFrame(preview_rows)
display(split_preview)

## Save the split summary

The text files are the main output, but I also want a small summary table saved with them so I can check the counts later without reopening everything.

In [ ]:
# ==============================================================================
# 4. SAVE THE SPLIT SUMMARY TABLES
# ==============================================================================
split_summary_path = os.path.join(SPLITS_DIR, 'split_summary.csv')
split_preview_path = os.path.join(SPLITS_DIR, 'split_preview.csv')

split_summary.to_csv(split_summary_path, index=False)
split_preview.to_csv(split_preview_path, index=False)

print(f'Split summary saved to: {split_summary_path}')
print(f'Split preview saved to: {split_preview_path}')

## GitHub sync note

Same idea as notebook `00`. The split files live in Drive. The notebook itself stays versioned in GitHub.

In [ ]:
# ==============================================================================
# 5. SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
import json

REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks', '01_data_splitting2.ipynb')
DRIVE_NOTEBOOK_PATH = '/content/drive/MyDrive/Colab Notebooks/01_data_splitting2.ipynb'

if os.path.exists(DRIVE_NOTEBOOK_PATH):
    !cp "{DRIVE_NOTEBOOK_PATH}" "{REPO_NOTEBOOK_PATH}"

    # Strip widget metadata if Colab adds it so GitHub rendering stays cleaner.
    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/01_data_splitting2.ipynb
    !git commit -m "Update 01_data_splitting2" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
else:
    print(f'Notebook file not found at: {DRIVE_NOTEBOOK_PATH}')
    print('Save the notebook in Colab, then run this cell again.')